# 1 Import module and data

In [1]:
import hashlib, urllib.request
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

URL = "https://docs.ambroise.cloud/velib/velib-2026-10-08.csv.gz"
FILE = "velib-2026-10-08.csv.gz"
MD5 = "02ab1e3fdac10579e4ce93d1540a5dc7"

path, _ = urllib.request.urlretrieve(URL, FILE)
with open(path, "rb") as f:
    assert hashlib.md5(f.read()).hexdigest() == MD5, "md5 mismatch"
print("MD5 OK")

MD5 OK


In [6]:
raw = pd.read_csv(path)
for c in ["collected_at_utc", "source_updated_at"]:
    raw[c] = pd.to_datetime(raw[c], utc=True)

df = raw.copy()   # raw is never modified
print("Dataset shape:", raw.shape)
print("Number of rows:", raw.shape[0])
print("Number of columns:", raw.shape[1])
print("------------------------------------------------------")
print("\n Column names:")
print(raw.columns.tolist())

Dataset shape: (329680, 16)
Number of rows: 329680
Number of columns: 16
------------------------------------------------------

 Column names:
['collected_at_utc', 'source_updated_at', 'station_code', 'name', 'commune', 'insee_code', 'latitude', 'longitude', 'capacity', 'bikes_available', 'mechanical_bikes', 'electric_bikes', 'docks_available', 'is_installed', 'is_renting', 'is_returning']


In [7]:
df = df[df["collected_at_utc"] != pd.Timestamp("2026-08-25 11:57:26+00:00")].copy()
df["slot"] = df["collected_at_utc"].dt.round("15min")
assert not df.duplicated(["station_code", "slot"]).any()

df = df.sort_values(["station_code", "slot"]).reset_index(drop=True)
print(len(df))
print(df.groupby("station_code").size().unique())

329600
[4120]


In [8]:
df["N"] = df["mechanical_bikes"] + df["electric_bikes"]
H = pd.Timedelta("30min")

future = df[["station_code", "slot", "N"]].copy()
future["slot"] = future["slot"] - H
future = future.rename(columns={"N": "N_future"})

lab = df.merge(future, on=["station_code", "slot"], how="left")
lab["y"] = (lab["N_future"] < 2).astype(float).where(lab["N_future"].notna())

print("rows:", len(lab))
print("rows without a t+30min row:", lab["N_future"].isna().sum())

rows: 329600
rows without a t+30min row: 2310


In [9]:
lab = lab.dropna(subset=["y"]).copy()
lab["y"] = lab["y"].astype(int)
print("labelled rows:", len(lab))
print("positive rate:", lab["y"].mean())

labelled rows: 327290
positive rate: 0.11187937303308992


### Row 6 feature

In [21]:
# lag features: take N from (same station, slot - k*15min), using df as the source
def add_lag(frame, k):
    past = df[["station_code", "slot", "N"]].copy()
    past["slot"] = past["slot"] + k * pd.Timedelta("15min")   # move the past row forward to time t
    past = past.rename(columns={"N": f"N_lag{k}"})
    return frame.merge(past, on=["station_code", "slot"], how="left")

feat = lab.copy()
feat = add_lag(feat, 1)
feat = add_lag(feat, 2)
feat["diff1"] = feat["N"] - feat["N_lag1"]

# calendar features in Paris local time
local = feat["slot"].dt.tz_convert("Europe/Paris")
feat["hour_sin"] = np.sin(2 * np.pi * (local.dt.hour + local.dt.minute / 60) / 24)
feat["hour_cos"] = np.cos(2 * np.pi * (local.dt.hour + local.dt.minute / 60) / 24)
feat["dow"] = local.dt.dayofweek

feat = feat.dropna(subset=["N_lag1", "N_lag2"]).copy()
print("rows after lags:", len(feat))
print("rows lost:", len(lab) - len(feat))
print(feat[["N", "N_lag1", "N_lag2", "diff1", "hour_sin", "hour_cos", "dow"]].head())


# longer past: 45 min and 60 min ago
feat = add_lag(feat, 3)
feat = add_lag(feat, 4)
feat = feat.dropna(subset=["N_lag3", "N_lag4"]).copy()

# summaries of the past hour (only from current and lagged values)
past_cols = ["N", "N_lag1", "N_lag2", "N_lag3", "N_lag4"]
feat["N_mean_1h"] = feat[past_cols].mean(axis=1)
feat["N_min_1h"]  = feat[past_cols].min(axis=1)
feat["diff4"]     = feat["N"] - feat["N_lag4"]          # trend over 1 hour

# current composition and station type
feat["is_paris"]   = (feat["insee_code"] == 75056).astype(int)
feat["weekend"]    = (feat["dow"] >= 5).astype(int)
feat["src_lag_min"] = (feat["collected_at_utc"] - feat["source_updated_at"]).dt.total_seconds() / 60
# mechanical_bikes and electric_bikes are already columns of feat

FEATS = ["N", "N_lag1", "N_lag2", "N_lag3", "N_lag4", "diff1", "diff4",
         "N_mean_1h", "N_min_1h", "mechanical_bikes", "electric_bikes",
         "is_paris", "weekend", "hour_sin", "hour_cos", "src_lag_min"]
print("rows:", len(feat), "| features:", len(FEATS))

rows after lags: 323460
rows lost: 3830
    N  N_lag1  N_lag2  diff1  hour_sin  hour_cos  dow
2  34    33.0    34.0    1.0 -0.608761 -0.793353    1
3  33    34.0    33.0   -1.0 -0.659346 -0.751840    1
4  34    33.0    34.0    1.0 -0.707107 -0.707107    1
5  35    34.0    33.0    1.0 -0.751840 -0.659346    1
6  36    35.0    34.0    1.0 -0.793353 -0.608761    1
rows: 320530 | features: 16


In [15]:
# longer past: 45 min and 60 min ago
feat = add_lag(feat, 3)
feat = add_lag(feat, 4)
feat = feat.dropna(subset=["N_lag3", "N_lag4"]).copy()

# summaries of the past hour (only from current and lagged values)
past_cols = ["N", "N_lag1", "N_lag2", "N_lag3", "N_lag4"]
feat["N_mean_1h"] = feat[past_cols].mean(axis=1)
feat["N_min_1h"]  = feat[past_cols].min(axis=1)
feat["diff4"]     = feat["N"] - feat["N_lag4"]          # trend over 1 hour

# current composition and station type
feat["is_paris"]   = (feat["insee_code"] == 75056).astype(int)
feat["weekend"]    = (feat["dow"] >= 5).astype(int)
feat["src_lag_min"] = (feat["collected_at_utc"] - feat["source_updated_at"]).dt.total_seconds() / 60
# mechanical_bikes and electric_bikes are already columns of feat

FEATS = ["N", "N_lag1", "N_lag2", "N_lag3", "N_lag4", "diff1", "diff4",
         "N_mean_1h", "N_min_1h", "mechanical_bikes", "electric_bikes",
         "is_paris", "weekend", "hour_sin", "hour_cos", "src_lag_min"]
print("rows:", len(feat), "| features:", len(FEATS))

rows: 320530 | features: 16


In [22]:
lost = lab.merge(feat[["station_code", "slot"]].assign(kept=1),
                 on=["station_code", "slot"], how="left")
lost = lost[lost["kept"].isna()]

print(lost.groupby("station_code").size().describe())   # lost rows per station
print(lost["slot"].dt.date.value_counts().head(10))     # which days lose the most

count    80.000000
mean     84.500000
std       1.948709
min      82.000000
25%      82.000000
50%      86.000000
75%      86.000000
max      86.000000
dtype: float64
slot
2026-09-08    900
2026-10-07    820
2026-09-02    740
2026-10-01    720
2026-09-14    690
2026-08-28    670
2026-09-19    640
2026-09-25    620
2026-08-25    320
2026-08-26    320
Name: count, dtype: int64


Rows lacking either lag (3,830, 1.2%) were removed; the loss is spread evenly across stations (46-49 rows each). No value was imputed.

### Row 7 divise train, val, test

In [23]:
local = feat["slot"].dt.tz_convert("Europe/Paris")
val_start  = pd.Timestamp("2026-09-25 00:00", tz="Europe/Paris")
test_start = pd.Timestamp("2026-10-02 00:00", tz="Europe/Paris")
embargo    = pd.Timedelta("1h")

train = feat[local < pd.Timestamp("2026-09-23 00:00", tz="Europe/Paris")]
val   = feat[(local >= val_start) & (local < test_start - embargo)]
test  = feat[local >= test_start]

for name, d in [("train", train), ("val", val), ("test", test)]:
    print(name, "| rows:", len(d),
          "| from", d["slot"].min(), "to", d["slot"].max(),
          "| positive rate:", round(d["y"].mean(), 4))

train | rows: 211270 | from 2026-08-25 13:00:00+00:00 to 2026-09-22 21:45:00+00:00 | positive rate: 0.1036
val | rows: 51220 | from 2026-09-24 22:00:00+00:00 to 2026-10-01 20:45:00+00:00 | positive rate: 0.1325
test | rows: 51720 | from 2026-10-01 22:00:00+00:00 to 2026-10-08 19:45:00+00:00 | positive rate: 0.1232


In [24]:
from sklearn.metrics import precision_score, recall_score, f1_score

for name, d in [("val", val), ("test", test)]:
    y = d["y"]
    pers = (d["N"] < 2).astype(int)          # persistence: future = current
    print(name,
          "| persistence P:", round(precision_score(y, pers), 3),
          "R:", round(recall_score(y, pers), 3),
          "F1:", round(f1_score(y, pers), 3))
    print(name, "| majority class: always 0, recall = 0")

val | persistence P: 0.761 R: 0.76 F1: 0.76
val | majority class: always 0, recall = 0
test | persistence P: 0.733 R: 0.735 F1: 0.734
test | majority class: always 0, recall = 0


# 3 Model Bayes

In [25]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import average_precision_score

FEATS = ["N", "N_lag1", "N_lag2", "diff1", "hour_sin", "hour_cos", "dow"]

nb = GaussianNB()
nb.fit(train[FEATS], train["y"])

p_val = nb.predict_proba(val[FEATS])[:, 1]
print("val AP:", round(average_precision_score(val["y"], p_val), 3),
      "| baseline AP (positive rate):", round(val["y"].mean(), 3))

for tau in [0.1, 0.2, 0.3, 0.5, 0.7]:
    pred = (p_val > tau).astype(int)
    print("tau", tau,
          "| P:", round(precision_score(val["y"], pred), 3),
          "R:", round(recall_score(val["y"], pred), 3),
          "F1:", round(f1_score(val["y"], pred), 3))

val AP: 0.651 | baseline AP (positive rate): 0.133
tau 0.1 | P: 0.384 R: 0.966 F1: 0.549
tau 0.2 | P: 0.392 R: 0.962 F1: 0.557
tau 0.3 | P: 0.406 R: 0.958 F1: 0.571
tau 0.5 | P: 0.424 R: 0.953 F1: 0.587
tau 0.7 | P: 0.44 R: 0.945 F1: 0.601


### Row 9

In [26]:
from sklearn.metrics import (confusion_matrix, precision_score, recall_score,
                             f1_score, fbeta_score, average_precision_score,
                             brier_score_loss, log_loss)

def report(name, y, p, tau=0.5, c_fn=5, c_fp=1):
    pred = (p > tau).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        "model": name, "tau": tau,
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred),
        "F1": f1_score(y, pred),
        "F2": fbeta_score(y, pred, beta=2),
        "AP": average_precision_score(y, p),
        "AP/pi": average_precision_score(y, p) / y.mean(),
        "Brier": brier_score_loss(y, p),
        "logloss": log_loss(y, np.clip(p, 1e-6, 1 - 1e-6)),
        "cost": (c_fp * fp + c_fn * fn) / len(y),   # c_fn/c_fp = 5 is a placeholder
    }

y = val["y"]
rows = [
    report("majority", y, np.zeros(len(val))),
    report("persistence", y, (val["N"] < 2).astype(float)),
    report("naive_bayes", y, nb.predict_proba(val[FEATS])[:, 1], tau=0.7),
]
table = pd.DataFrame(rows).set_index("model").round(3)
pd.set_option("display.width", 250)
print(table.T)   # models as columns: easier to read

model       majority  persistence  naive_bayes
tau            0.500        0.500        0.700
TP             0.000     5159.000     6418.000
FP             0.000     1621.000     8156.000
FN          6788.000     1629.000      370.000
TN         44432.000    42811.000    36276.000
precision      0.000        0.761        0.440
recall         0.000        0.760        0.945
F1             0.000        0.760        0.601
F2             0.000        0.760        0.769
AP             0.133        0.610        0.651
AP/pi          1.000        4.604        4.909
Brier          0.133        0.063        0.158
logloss        1.831        0.877        0.678
cost           0.663        0.191        0.195
